# Regional two-reference drift and prediction skill

This notebook reads the compact multi-region products ensured via `workflows.diagnostics.drift_inputs` (or precomputed by `preprocessing/drift/0_run_drift_diag.ipynb`). It produces regional drift and regional prediction-skill figures for Niño3.4 and the North Atlantic, while retaining the global-land H2OSOI diagnostic. Global maps are handled by `5a_refactor_drift_map.ipynb`; regime-frequency, spatial-summary, and drift–skill relationship figures are intentionally outside this notebook.
Input preparation is automatic with `DRIFT_INPUT_MODE="auto"`; the `preprocessing/drift/0_run_drift_input.ipynb` and `0_run_drift_diag.ipynb` notebooks are optional batch drivers. Raw hindcast, observation, and historical reference archives must be available when inputs need building.


In [ ]:
import os
import sys
from pathlib import Path

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = ([Path(_repo_override).expanduser().resolve()] if _repo_override
                    else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
REPO_ROOT = next((p for p in _repo_candidates
                 if (p / "workflows" / "diagnostics" / "drift_inputs.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.drift_inputs import (
    DEFAULT_OUTPUT_ROOT, DEFAULT_FIGURE_ROOT, DEFAULT_VARIABLES,
    DEFAULT_INIT_MONTHS, DEFAULT_SOURCES, DEFAULT_SOURCE_LABELS,
    DEFAULT_SOURCE_COLORS, DEFAULT_PLOT_REGIONS, DEFAULT_REGION_LABELS,
    DEFAULT_REGION_FILE_LABELS, MONTH_NAMES,
    load_regional_manifest, regional_product_path, figure_size
)

%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import xarray as xr


## 1. Configuration

In [ ]:
OUTPUT_ROOT = DEFAULT_OUTPUT_ROOT
FIGURE_ROOT = DEFAULT_FIGURE_ROOT
VARIABLES = DEFAULT_VARIABLES
INIT_MONTHS = DEFAULT_INIT_MONTHS
SOURCES = DEFAULT_SOURCES
SOURCE_LABELS = DEFAULT_SOURCE_LABELS
SOURCE_COLORS = DEFAULT_SOURCE_COLORS
PLOT_REGIONS = DEFAULT_PLOT_REGIONS
REGION_LABELS = DEFAULT_REGION_LABELS
REGION_FILE_LABELS = DEFAULT_REGION_FILE_LABELS
DRIFT_INPUT_MODE = os.environ.get('DRIFT_INPUT_MODE', 'auto')

DRIFT_METRICS = (
    ('e_obs', 'Observation departure ($e_{\\mathrm{obs}}$)'),
    ('e_att', 'Attractor departure ($e_{\\mathrm{att}}$)'),
    ('delta_abs', 'Absolute distance change ($\\Delta|e|$)'),
)
SKILL_METRICS = (
    ('skill_rmse', 'RMSE'),
    ('skill_ensemble_spread', 'Ensemble spread'),
)

FIGURE_SCALE = 1.0
FIGURE_DPI = 300
FIGURE_PREFIX = 'fig_two_ref'
SHOW_FIGURES_INLINE = True
FONT_SIZE = 9.0
TITLE_FONT_SIZE = 10.0
LEGEND_FONT_SIZE = 8.5
FIGURE_SIZE = (12.0, 7.5)

def get_figure_size():
    return figure_size(FIGURE_SIZE[0], FIGURE_SIZE[1], scale=FIGURE_SCALE)

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Validate and load regional products

Only source-specific `regional` products are required. Unrelated spatial, paired-regime, and other diagnostic products are neither opened nor required for validation.

In [ ]:
regional_manifest = load_regional_manifest(
    OUTPUT_ROOT, VARIABLES, INIT_MONTHS, SOURCES, PLOT_REGIONS,
    mode=DRIFT_INPUT_MODE,
)
display(regional_manifest)

def load_regional_product(variable, init_month, source):
    path = regional_product_path(variable, init_month, source, output_root=OUTPUT_ROOT)
    with xr.open_dataset(path) as opened:
        saved = opened.load()
    required_drift = {name for name, _ in DRIFT_METRICS}
    required_skill = {f'skill_{name}' for name, _ in SKILL_METRICS}
    missing = (required_drift | required_skill) - set(saved.data_vars)
    if missing:
        raise KeyError(f'{path} is missing regional fields: {sorted(missing)}')
    expected_regions = PLOT_REGIONS[variable]
    if 'region' not in saved.dims:
        raise ValueError(f'{path} predates multi-region output; rerun input preparation')
    available_regions = tuple(saved.region.values.astype(str))
    missing_regions = set(expected_regions) - set(available_regions)
    if missing_regions:
        raise ValueError(f'{path} is missing regions: {sorted(missing_regions)}')
    regional = saved[sorted(required_drift)]
    skill = saved[sorted(required_skill)].rename(
        {name: name.removeprefix('skill_') for name in required_skill}
    )
    return {'regional': regional, 'skill': skill, 'path': path}

results = {
    (variable, init_month, source): load_regional_product(variable, init_month, source)
    for variable in VARIABLES
    for init_month in INIT_MONTHS
    for source in SOURCES
}
print(f'Loaded {len(results)} source-specific regional products.')


## 3. Regional drift figures

The first two columns are signed departures. For the two absolute-distance-change columns, negative values indicate movement closer to the reference and positive values indicate movement farther away.

In [ ]:
figure_paths = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(INIT_MONTHS), len(DRIFT_METRICS),
            figsize=get_figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, init_month in enumerate(INIT_MONTHS):
            for col, (metric, title) in enumerate(DRIFT_METRICS):
                ax = axes[row, col]
                for source in SOURCES:
                    field = results[(
                        variable, init_month, source
                    )]['regional'][metric].sel(region=region_name)
                    field.mean('Y', skipna=True).plot(
                        ax=ax, label=SOURCE_LABELS[source],
                        color=SOURCE_COLORS[source],
                    )
                ax.axhline(0, color='0.35', linewidth=0.8)
                ax.set_title(f'{MONTH_NAMES[init_month]}: {title}')
                ax.set_xlabel('Lead month')
                ax.legend()
        fig.suptitle(
            f'{variable}: regional two-reference drift, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_regional_drift.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        figure_paths.append(path)

## 4. Regional prediction-skill figures

In [ ]:
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(INIT_MONTHS), len(SKILL_METRICS),
            figsize=get_figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, init_month in enumerate(INIT_MONTHS):
            for col, (metric, title) in enumerate(SKILL_METRICS):
                ax = axes[row, col]
                for source in SOURCES:
                    field = results[(
                        variable, init_month, source
                    )]['skill'][metric].sel(region=region_name)
                    field.plot(
                        ax=ax, label=SOURCE_LABELS[source],
                        color=SOURCE_COLORS[source],
                    )
                ax.set_title(f'{MONTH_NAMES[init_month]}: {title}')
                ax.set_xlabel('Lead month')
                ax.legend()
        fig.suptitle(
            f'{variable}: regional prediction skill, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_regional_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        figure_paths.append(path)

## 5. Validation

In [ ]:
expected_figure_count = 2 * sum(
    len(PLOT_REGIONS[variable]) for variable in VARIABLES
)
assert len(figure_paths) == expected_figure_count
assert all(path.is_file() and path.stat().st_size > 0 for path in figure_paths)
assert len(results) == len(VARIABLES) * len(INIT_MONTHS) * len(SOURCES)
for (variable, _, _), result in results.items():
    available = set(result['regional'].region.values.astype(str))
    assert set(PLOT_REGIONS[variable]) <= available
print(
    f'Validated {len(figure_paths)} regional drift/skill figures from '
    f'{len(results)} source-specific regional products.'
)